![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [346]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

In [347]:
from transformers import pipeline

pipe = pipeline('sentiment-analysis', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english')

Device set to use cpu


In [348]:
data = pd.read_csv('data/car_reviews.csv', delimiter=';')
data

,Review,Class
0,I am very satisfied with my 2014 Nissan NV SL....,POSITIVE
1,The car is fine. It's a bit loud and not very ...,NEGATIVE
2,"My first foreign car. Love it, I would buy ano...",POSITIVE
3,I've come across numerous reviews praising the...,NEGATIVE
4,I've been dreaming of owning an SUV for quite ...,POSITIVE


In [349]:
reviews = data['Review'].tolist()
reviews

['I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use. Camping, road trips, etc. We dont have any children so I store most of the seats in my warehouse. I wanted the passenger van for the rear air conditioning. We drove our van from Florida to California for a Cross Country trip in 2014. We averaged about 18 mpg. We drove thru a lot of rain and It was a very comfortable and stable vehicle. The V8 Nissan Titan engine is a 500k mile engine. It has been tested many times by delivery and trucking companies. This is why Nissan gives you a 5 year or 100k mile bumper to bumper warranty. Many people are scared about driving this van because of its size. But with front and rear sonar sensors, large mirrors and the back up camera. It is easy to drive. The front and rear sensors also monitor the front and rear sides of the bumpers making it easier to park close to objects. Our Nissan NV is a Tow Monster. It pulls our 5000 pound travel trailer 

In [350]:
real_labels = data['Class'].tolist()
real_labels

['POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE']

In [351]:
references = [1 if label == 'POSITIVE' else 0 for label in real_labels]
references

[1, 0, 1, 0, 1]

In [352]:
predicted_labels = pipe(reviews)
predicted_labels

[{'label': 'POSITIVE', 'score': 0.929397702217102},
 {'label': 'POSITIVE', 'score': 0.8654273152351379},
 {'label': 'POSITIVE', 'score': 0.9994640946388245},
 {'label': 'NEGATIVE', 'score': 0.9935314059257507},
 {'label': 'POSITIVE', 'score': 0.9986565113067627}]

In [353]:
predicted_labels[0]['label']

'POSITIVE'

In [354]:
predictions = []
for i in range(len(predicted_labels)):
    if predicted_labels[i]['label'] == 'POSITIVE':
        predictions.append(1)
    else:
        predictions.append(0)

predictions

[1, 1, 1, 0, 1]

In [355]:
import evaluate

In [356]:
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

In [357]:
accuracy_result = accuracy.compute(predictions=predictions, references=references)
accuracy_result = accuracy_result['accuracy']
f1_result = f1.compute(predictions=predictions, references=references)
f1_result = f1_result['f1']
print('Accuracy:', accuracy_result)
print('F1:', f1_result)

Accuracy: 0.8
F1: 0.8571428571428571


In [358]:
pipe = pipeline("translation", 
                model="Helsinki-NLP/opus-mt-tc-big-en-es")

Device set to use cpu


In [359]:
translated_review = pipe(reviews[0], max_length=30)[0]['translation_text']
translated_review

Your input_length: 358 is bigger than 0.9 * max_length: 30. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


'Estoy muy satisfecho con mi 2014 Nissan NV. Tengo esta camioneta para mis entregas de negocios y uso personal. La cabina del Nissan NV.'

In [360]:
with open('data/reference_translations.txt', 'r') as f:
    texts = f.readlines()

reference_text = [text.strip() for text in texts]
reference_text

['Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.',
 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.']

In [361]:
print(repr(translated_review))
print(repr(reference_text))

'Estoy muy satisfecho con mi 2014 Nissan NV. Tengo esta camioneta para mis entregas de negocios y uso personal. La cabina del Nissan NV.'
['Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.', 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.']


In [362]:
bleu = evaluate.load('bleu')

In [363]:
bleu_score = bleu.compute(
    predictions=[translated_review],   
    references=[reference_text]   
)
bleu_score['bleu']

0.378448113759187

In [364]:
model_name = "deepset/minilm-uncased-squad2"
qa = pipeline('question-answering',
             model=model_name,
              tokenizer=model_name
             )

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [365]:
question = 'What did he like about the brand?'
context = reviews[1]
qa_input = {
    'question': question,
    'context': context
}
#print(reviews[1])

answer = qa(qa_input)['answer']
print('Answer:', answer)

Answer: ride quality, reliability


In [366]:
summarizer = pipeline('summarization', 
                      model='google-t5/t5-small',
                     max_length=55)

Device set to use cpu


In [367]:
summary = summarizer(reviews[-1])
summarized_text = summary[0]['summary_text']
summarized_text

'the Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment . the financial arrangement is quite reasonable . I find myself needing extra caution when making lane changes .'